# 수요 예측 프로젝트 - 데이터 분석 (EDA)

## 이 노트북에서 배울 것

| 단계 | 내용 |
|------|------|
| 1-1 | 데이터 로드 |
| 1-2 | 기본 구조 파악 |
| 1-3 | 결측값 확인 및 처리 |
| 1-4 | 시각화 (트렌드, 비교, 패턴) |

**데이터**: 제조업체 제품 수요 이력 (2011~2017년, 약 100만 행)

---
## 1-1. 데이터 로드

`pd.read_csv()` 로 CSV 파일을 읽어서 DataFrame으로 만든다.

DataFrame = 엑셀 시트처럼 행(row)과 열(column)로 구성된 표

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')                    # 경고 메시지 숨기기
matplotlib.rcParams['font.family'] = 'Malgun Gothic' # 한글 폰트 설정
matplotlib.rcParams['axes.unicode_minus'] = False    # 마이너스 기호 깨짐 방지

print('라이브러리 로드 완료')

In [ ]:
# CSV 파일 읽기
# 파일 경로는 이 노트북 기준 상대경로
df_raw = pd.read_csv('./archive/Historical Product Demand.csv')

print(f'데이터 로드 완료: {len(df_raw):,}행')

---
## 1-2. 기본 구조 파악

데이터가 어떻게 생겼는지 먼저 파악해야 어떻게 분석할지 계획을 세울 수 있다.

In [ ]:
# 상위 5행 확인 - 데이터가 어떻게 생겼는지 눈으로 확인
df_raw.head()

In [ ]:
# 컬럼 이름, 데이터 타입, 결측치 개수 확인
# object = 문자열, int64 = 정수, float64 = 소수
df_raw.info()

In [ ]:
# 행 수, 열 수 확인
rows, cols = df_raw.shape
print(f'행(row) 수: {rows:,}개  ← 데이터 건수')
print(f'열(col) 수: {cols}개   ← 컬럼(변수) 수')
print()
print('컬럼 목록:', list(df_raw.columns))

In [ ]:
# 숫자형 컬럼의 통계 요약
# count=건수, mean=평균, std=표준편차, min/max=최솟/최댓값
# 지금은 Order_Demand가 문자열(object)이라 나오지 않음
df_raw.describe()

In [ ]:
# 각 컬럼의 고유값 수 확인
print('=== 고유값 수 ===')
print(f'제품 종류: {df_raw["Product_Code"].nunique():,}개')
print(f'창고 수:   {df_raw["Warehouse"].nunique()}개 → {df_raw["Warehouse"].unique()}')
print(f'카테고리:  {df_raw["Product_Category"].nunique()}개')
print()
# Order_Demand 샘플 확인 (왜 문자열인지)
print('=== Order_Demand 샘플 (이상한 형식 확인) ===')
print(df_raw['Order_Demand'].head(20).tolist())

---
## 1-3. 결측값 확인 및 처리

**결측값(NaN)** = 비어 있는 값. 그대로 두면 모델이 오류를 낸다.

이 데이터의 특이한 문제:
- `Date`: 일부 결측 → 삭제
- `Order_Demand`: 숫자인데 문자열 형태 + `(500)` 같은 이상한 형식 → 변환 필요

In [ ]:
# 컬럼별 결측값 개수 확인
null_counts = df_raw.isnull().sum()
print('=== 결측값 개수 ===')
print(null_counts)
print()
# 결측값 비율
print('=== 결측값 비율 ===')
print((null_counts / len(df_raw) * 100).round(2).astype(str) + '%')

In [ ]:
# Order_Demand 정제 함수
# 실제 데이터는 더럽다 → 이걸 처리하는 게 실력
def clean_demand(val):
    val = str(val).strip()                        # 앞뒤 공백 제거
    if val.startswith('(') and val.endswith(')'): # "(500)" 형태 = 반품/취소
        try:
            return -float(val[1:-1].replace(',', ''))  # 음수로 변환
        except:
            return np.nan
    try:
        return float(val.replace(',', ''))         # 일반 숫자
    except:
        return np.nan                              # 변환 실패 → 결측치

# 데이터 복사 후 전처리 (원본 보존)
df = df_raw.copy()
df['Order_Demand'] = df['Order_Demand'].apply(clean_demand)
df['Date']         = pd.to_datetime(df['Date'], errors='coerce')  # 날짜형으로 변환

# 결측치 및 음수 제거
before = len(df)
df = df.dropna(subset=['Date', 'Order_Demand'])   # 날짜·수요 결측 행 삭제
df = df[df['Order_Demand'] > 0]                   # 음수(반품) 제거
after = len(df)

print(f'전처리 전: {before:,}행')
print(f'전처리 후: {after:,}행')
print(f'제거된 행: {before - after:,}행')

In [ ]:
# 날짜에서 파생 변수 추출 (Feature Engineering)
# 날짜 문자열 자체는 모델이 이해 못함 → 숫자로 분해
df['Year']      = df['Date'].dt.year     # 연도: 2011, 2012 ...
df['Month']     = df['Date'].dt.month    # 월: 1~12
df['Quarter']   = df['Date'].dt.quarter  # 분기: 1~4
df['YearMonth'] = df['Date'].dt.to_period('M')  # 2011-01 형태

print('파생 변수 추가 완료')
df.head(3)

In [ ]:
# 전처리 후 Order_Demand 통계 요약
print('=== Order_Demand 통계 요약 ===')
print(df['Order_Demand'].describe())
print()
print(f'평균:   {df["Order_Demand"].mean():>12,.1f}')
print(f'중앙값: {df["Order_Demand"].median():>12,.1f}')
print(f'최댓값: {df["Order_Demand"].max():>12,.1f}')

---
## 1-4. 시각화

숫자만 보면 패턴이 안 보인다. **그래프로 봐야 이상한 점, 트렌드, 계절성이 보인다.**

- 날짜별 수요량 그래프
- 제품 카테고리별 수요량 비교
- 창고별 비교
- 월별 패턴 (계절성)

In [ ]:
# ── 그래프 1: 전체 월별 수요량 추이 ──
# 월별로 집계해서 큰 흐름 확인
monthly_total = (
    df.groupby('YearMonth')['Order_Demand']
    .sum()
    .reset_index()
)
monthly_total['date'] = monthly_total['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_total['date'], monthly_total['Order_Demand'],
        color='steelblue', linewidth=1.5)
ax.fill_between(monthly_total['date'], monthly_total['Order_Demand'],
                alpha=0.2, color='steelblue')
ax.set_title('전체 월별 총 주문량 추이 (2011~2017)', fontsize=14)
ax.set_xlabel('날짜')
ax.set_ylabel('총 주문량')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ★ 보는 포인트: 전체적인 증가/감소 트렌드, 급등·급락 구간

In [ ]:
# ── 그래프 2: 카테고리별 총 수요량 비교 (상위 10개) ──
cat_total = (
    df.groupby('Product_Category')['Order_Demand']
    .sum()
    .sort_values(ascending=False)  # 내림차순 정렬
    .head(10)                      # 상위 10개만
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(cat_total['Product_Category'],
               cat_total['Order_Demand'],
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(cat_total))))

# 막대 끝에 수치 표시
for bar, val in zip(bars, cat_total['Order_Demand']):
    ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9)

ax.set_title('카테고리별 총 주문량 (상위 10개)', fontsize=14)
ax.set_xlabel('총 주문량')
ax.invert_yaxis()  # 1위를 위에 표시
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ★ 보는 포인트: 어떤 카테고리가 수요가 많은지 → 생산 우선순위

In [ ]:
# ── 그래프 3: 창고별 수요량 비교 (파이차트 + 막대) ──
wh_total = df.groupby('Warehouse')['Order_Demand'].sum().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 파이차트: 비율 확인
ax1.pie(wh_total['Order_Demand'],
        labels=wh_total['Warehouse'],
        autopct='%1.1f%%',
        colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax1.set_title('창고별 수요 비율')

# 막대차트: 절댓값 확인
ax2.bar(wh_total['Warehouse'], wh_total['Order_Demand'],
        color=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax2.set_title('창고별 총 주문량')
ax2.set_xlabel('창고')
ax2.set_ylabel('총 주문량')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ★ 보는 포인트: 어느 창고에 물량이 집중되는지

In [ ]:
# ── 그래프 4: 월별 패턴 (계절성 확인) ──
monthly_pattern = (
    df.groupby('Month')['Order_Demand']
    .mean()  # 월별 평균 (연도 관계없이)
    .reset_index()
)
month_labels = ['1월','2월','3월','4월','5월','6월',
                '7월','8월','9월','10월','11월','12월']

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(1, 13), monthly_pattern['Order_Demand'],
              color=plt.cm.RdYlGn(monthly_pattern['Order_Demand'] /
                                   monthly_pattern['Order_Demand'].max()))
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.axhline(monthly_pattern['Order_Demand'].mean(),
           color='red', linestyle='--', linewidth=1.2,
           label=f'연평균 ({monthly_pattern["Order_Demand"].mean():,.0f})')
ax.set_title('월별 평균 주문량 (계절성 패턴)', fontsize=14)
ax.set_ylabel('평균 주문량')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ★ 보는 포인트: 특정 달에 수요가 몰리는지 → 사전 생산 계획 필요

In [ ]:
# ── 그래프 5: 연도×월 히트맵 (계절성 + 연도별 변화 동시 확인) ──
pivot = df.pivot_table(
    values='Order_Demand',
    index='Year',
    columns='Month',
    aggfunc='sum'
)
pivot.columns = month_labels

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt='.0f',
            cmap='YlOrRd',        # 노랑→주황→빨강 (수요 많을수록 진함)
            linewidths=0.5, ax=ax)
ax.set_title('연도별 × 월별 총 주문량 히트맵', fontsize=14)
ax.set_ylabel('연도')
ax.set_xlabel('월')
plt.tight_layout()
plt.show()

# ★ 보는 포인트:
# - 가로(행): 연도별로 수요가 증가하는 추세인지
# - 세로(열): 특정 달이 매년 높은지 (계절성)
# - 진한 칸: 해당 월/연도에 생산 집중 필요

In [ ]:
# ── 그래프 6: 박스플롯으로 이상치 확인 ──
# 박스플롯: 중앙값, 사분위범위, 이상치를 한번에 보여줌
fig, ax = plt.subplots(figsize=(14, 4))

monthly_by_month = [
    df[df['Month'] == m]['Order_Demand'].values
    for m in range(1, 13)
]
bp = ax.boxplot(monthly_by_month, labels=month_labels, patch_artist=True)

# 박스 색상 설정
for patch in bp['boxes']:
    patch.set_facecolor('lightsteelblue')

ax.set_title('월별 주문량 분포 (박스플롯)', fontsize=14)
ax.set_xlabel('월')
ax.set_ylabel('주문량')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ★ 보는 포인트:
# - 박스가 높으면 → 그 달 수요 편차 크다 (예측 어려움)
# - 점(•)이 많으면 → 이상치 많다 (특별 이벤트/오류 데이터)
# - 박스 위치가 높은 달 → 수요 많은 달

---
## EDA 요약

| 항목 | 내용 |
|------|------|
| 데이터 크기 | 약 100만 행 × 5컬럼 |
| 기간 | 2011 ~ 2017년 |
| 주요 문제 | Order_Demand 문자열 형식, Date 결측 11,239건 |
| 카테고리 | 33개 (Category_001 ~ 033) |
| 창고 | 4곳 (Whse_A/C/J/S) |

### 다음 단계
- 월별 집계 후 LSTM으로 시계열 예측
- 카테고리별 안전재고 계산
- Streamlit 대시보드로 시각화